# antiSMASH 与 Prokka CDS 注释比较

本 Notebook 演示如何比较 antiSMASH 和 Prokka 输出的 CDS（编码序列）注释。

## 功能特性

1. **提取 CDS 注释** - 从 GenBank 文件中提取所有 CDS 特征
2. **位置比较** - 基于基因组位置匹配 CDS（允许容差）
3. **统计分析** - 计算匹配率和差异
4. **详细报告** - 生成可视化报告和 CSV 导出

## 使用场景

- 验证两个注释工具的一致性
- 识别工具特异性的注释
- 质量控制和结果验证

## 1. 环境准备

In [ ]:
# 环境设置from pathlib import Path# 创建工作目录WORK_DIR = Path('cds_comparison_runs')WORK_DIR.mkdir(exist_ok=True, parents=True)print(f"✓ 工作目录: {WORK_DIR}")

In [ ]:
# CDS 注释比较工具 - 内联版本from pathlib import Pathfrom typing import List, Dictfrom Bio import SeqIOfrom Bio.SeqFeature import SeqFeaturefrom collections import defaultdictclass CDSFeature:    """CDS 特征的简化表示"""        def __init__(self, feature: SeqFeature, record_id: str, source: str):        self.record_id = record_id        self.source = source        self.start = int(feature.location.start)        self.end = int(feature.location.end)        self.strand = feature.location.strand        self.length = self.end - self.start                # 提取注释信息        q = feature.qualifiers        self.gene = q.get('gene', [''])[0]        self.locus_tag = q.get('locus_tag', [''])[0]        self.product = q.get('product', [''])[0]        self.protein_id = q.get('protein_id', [''])[0]        self.translation = q.get('translation', [''])[0]        def __repr__(self):        return f"{self.source}:{self.record_id}[{self.start}:{self.end}]({self.gene or self.locus_tag})"        def matches_location(self, other: 'CDSFeature', tolerance: int = 10) -> bool:        """检查位置是否匹配（允许一定容差）"""        if self.strand != other.strand:            return False        return (abs(self.start - other.start) <= tolerance and                abs(self.end - other.end) <= tolerance)def extract_cds_from_gbk(gbk_path: Path, source: str = 'unknown') -> List[CDSFeature]:    """从 GenBank 文件提取 CDS 特征"""    cds_features = []        try:        for record in SeqIO.parse(str(gbk_path), 'genbank'):            for feature in record.features:                if feature.type == 'CDS':                    cds_features.append(CDSFeature(feature, record.id, source))        print(f"从 {gbk_path.name} 提取了 {len(cds_features)} 个 CDS")    except Exception as e:        print(f"解析 {gbk_path} 失败: {e}")        return cds_featuresdef extract_cds_from_dir(directory: Path, source: str = 'unknown') -> List[CDSFeature]:    """从目录中的所有 GenBank 文件提取 CDS"""    all_cds = []    gbk_files = list(directory.glob('**/*.gbk')) + list(directory.glob('**/*.gbff'))        for gbk_file in gbk_files:        cds_list = extract_cds_from_gbk(gbk_file, source)        all_cds.extend(cds_list)        print(f"从 {directory} 总共提取了 {len(all_cds)} 个 CDS (来自 {len(gbk_files)} 个文件)")    return all_cdsdef compare_cds(antismash_cds: List[CDSFeature],                prokka_cds: List[CDSFeature],                tolerance: int = 10) -> Dict:    """比较两组 CDS 注释"""    # 按 record_id 分组    as_by_record = defaultdict(list)    pk_by_record = defaultdict(list)        for cds in antismash_cds:        as_by_record[cds.record_id].append(cds)    for cds in prokka_cds:        pk_by_record[cds.record_id].append(cds)        all_records = set(as_by_record.keys()) | set(pk_by_record.keys())        # 比较结果    matched_pairs = []    antismash_only = []    prokka_only = []        for record_id in all_records:        as_cds = as_by_record.get(record_id, [])        pk_cds = pk_by_record.get(record_id, [])                # 找到匹配的 CDS        matched_as = set()        matched_pk = set()                for i, as_feature in enumerate(as_cds):            for j, pk_feature in enumerate(pk_cds):                if as_feature.matches_location(pk_feature, tolerance):                    matched_pairs.append((as_feature, pk_feature))                    matched_as.add(i)                    matched_pk.add(j)                # 只在 antiSMASH 中的        for i, as_feature in enumerate(as_cds):            if i not in matched_as:                antismash_only.append(as_feature)                # 只在 Prokka 中的        for j, pk_feature in enumerate(pk_cds):            if j not in matched_pk:                prokka_only.append(pk_feature)        return {        'matched': matched_pairs,        'antismash_only': antismash_only,        'prokka_only': prokka_only,        'summary': {            'total_matched': len(matched_pairs),            'total_antismash_only': len(antismash_only),            'total_prokka_only': len(prokka_only),            'total_antismash': len(antismash_cds),            'total_prokka': len(prokka_cds),        }    }def print_comparison_report(results: Dict):    """打印比较报告"""    summary = results['summary']        print("=" * 60)    print("CDS 注释比较报告")    print("=" * 60)    print(f"\n总计:")    print(f"  antiSMASH CDS: {summary['total_antismash']}")    print(f"  Prokka CDS: {summary['total_prokka']}")    print(f"\n匹配结果:")    print(f"  匹配的 CDS: {summary['total_matched']}")    print(f"  仅 antiSMASH: {summary['total_antismash_only']}")    print(f"  仅 Prokka: {summary['total_prokka_only']}")    print("=" * 60)def compare_annotations(antismash_path, prokka_path, tolerance=10, verbose=True):    """便捷函数：比较 antiSMASH 和 Prokka 注释"""    antismash_path = Path(antismash_path)    prokka_path = Path(prokka_path)        # 提取 CDS    if antismash_path.is_dir():        antismash_cds = extract_cds_from_dir(antismash_path, 'antismash')    else:        antismash_cds = extract_cds_from_gbk(antismash_path, 'antismash')        if prokka_path.is_dir():        prokka_cds = extract_cds_from_dir(prokka_path, 'prokka')    else:        prokka_cds = extract_cds_from_gbk(prokka_path, 'prokka')        # 比较    results = compare_cds(antismash_cds, prokka_cds, tolerance)        if verbose:        print_comparison_report(results)        return resultsdef export_comparison_to_csv(results: Dict, output_prefix: Path):    """导出比较结果到 CSV 文件"""    import pandas as pd        output_prefix = Path(output_prefix)    output_prefix.parent.mkdir(parents=True, exist_ok=True)        # 导出匹配的 CDS    if results['matched']:        matched_data = []        for as_cds, pk_cds in results['matched']:            matched_data.append({                'record_id': as_cds.record_id,                'start': as_cds.start,                'end': as_cds.end,                'as_gene': as_cds.gene,                'pk_gene': pk_cds.gene,                'as_product': as_cds.product,                'pk_product': pk_cds.product,            })        df = pd.DataFrame(matched_data)        df.to_csv(f"{output_prefix}_matched.csv", index=False)        # 导出仅在 antiSMASH 中的    if results['antismash_only']:        as_data = [{'record_id': cds.record_id, 'start': cds.start, 'end': cds.end,                    'gene': cds.gene, 'product': cds.product}                   for cds in results['antismash_only']]        df = pd.DataFrame(as_data)        df.to_csv(f"{output_prefix}_antismash_only.csv", index=False)        # 导出仅在 Prokka 中的    if results['prokka_only']:        pk_data = [{'record_id': cds.record_id, 'start': cds.start, 'end': cds.end,                    'gene': cds.gene, 'product': cds.product}                   for cds in results['prokka_only']]        df = pd.DataFrame(pk_data)        df.to_csv(f"{output_prefix}_prokka_only.csv", index=False)        print(f"✓ 结果已导出到 {output_prefix}_*.csv")print("✓ CDS 比较工具已加载")

In [ ]:
# 示例：使用单个 GenBank 文件
antismash_gbk = Path('esm3_pipeline/outputs/antismash_results/your_genome.gbk')
prokka_gbk = Path('prokka_esm3_runs/run_001/prokka_output/PROKKA.gbk')

# 检查文件是否存在
if antismash_gbk.exists():
    print(f"✅ antiSMASH 文件存在: {antismash_gbk}")
else:
    print(f"⚠️  antiSMASH 文件不存在: {antismash_gbk}")

if prokka_gbk.exists():
    print(f"✅ Prokka 文件存在: {prokka_gbk}")
else:
    print(f"⚠️  Prokka 文件不存在: {prokka_gbk}")

### 方法 B: 指定输出目录（自动查找 .gbk 文件）

In [ ]:
# 示例：使用输出目录
antismash_dir = Path('esm3_pipeline/outputs/antismash_results')
prokka_dir = Path('prokka_esm3_runs/run_001/prokka_output')

print(f"antiSMASH 目录: {antismash_dir}")
print(f"Prokka 目录: {prokka_dir}")

## 3. 快速比较（推荐方式）

使用便捷函数一步完成比较：

In [ ]:
# 方法 A：比较单个文件
results = compare_annotations(
    antismash_path=antismash_gbk,
    prokka_path=prokka_gbk,
    tolerance=10,  # 位置容差（碱基对）
    verbose=True   # 显示详细报告
    )

In [ ]:
# 方法 B：比较目录（推荐用于多个文件）
results = compare_annotations(
    antismash_path=antismash_dir,
    prokka_path=prokka_dir,
    tolerance=10,
    verbose=True
    )

## 4. 分步比较（高级用法）

如果需要更多控制，可以分步执行：

In [ ]:
# 步骤 1: 提取 CDS 注释
print("正在提取 antiSMASH CDS...")
antismash_cds = extract_cds_from_gbk(antismash_gbk, 'antismash')
# 或者从目录提取: antismash_cds = extract_cds_from_dir(antismash_dir, 'antismash')

print("\n正在提取 Prokka CDS...")
prokka_cds = extract_cds_from_gbk(prokka_gbk, 'prokka')
# 或者从目录提取: prokka_cds = extract_cds_from_dir(prokka_dir, 'prokka')

print(f"\n提取完成:")
print(f"  antiSMASH: {len(antismash_cds)} 个 CDS")
print(f"  Prokka: {len(prokka_cds)} 个 CDS")

In [ ]:
# 步骤 2: 执行比较
results = compare_cds(
    antismash_cds=antismash_cds,
    prokka_cds=prokka_cds,
    tolerance=10  # 位置匹配容差（碱基对）
    )

In [ ]:
# 步骤 3: 打印报告
print_comparison_report(results, verbose=True)

## 5. 查看详细结果

In [ ]:
# 查看匹配的 CDS 对（前 5 个）
print("匹配的 CDS 对示例:")
print("=" * 80)
for i, (as_cds, pk_cds) in enumerate(results['matched'][:5], 1):
    print(f"\n#{i}:")
    print(f"  位置: {as_cds.start}-{as_cds.end} ({as_cds.length} bp)")
    print(f"  antiSMASH: {as_cds.gene or as_cds.locus_tag} | {as_cds.product[:60]}")
    print(f"  Prokka:    {pk_cds.gene or pk_cds.locus_tag} | {pk_cds.product[:60]}")
    if as_cds.translation and pk_cds.translation:
        same_seq = "✅ 相同" if as_cds.translation == pk_cds.translation else "⚠️  不同"
        print(f"  蛋白序列: {same_seq}")

In [ ]:
# 查看仅在 antiSMASH 中的 CDS（前 5 个）
if results['antismash_only']:
    print("\n仅在 antiSMASH 中的 CDS:")
    print("=" * 80)
    for i, cds in enumerate(results['antismash_only'][:5], 1):
        print(f"#{i}: {cds.start:8d}-{cds.end:8d} ({cds.length:5d}bp) {cds.gene or cds.locus_tag:20s} {cds.product[:40]}")
else:
    print("\n✅ 所有 antiSMASH CDS 都在 Prokka 中找到了匹配")

In [ ]:
# 查看仅在 Prokka 中的 CDS（前 5 个）
if results['prokka_only']:
    print("\n仅在 Prokka 中的 CDS:")
    print("=" * 80)
    for i, cds in enumerate(results['prokka_only'][:5], 1):
        print(f"#{i}: {cds.start:8d}-{cds.end:8d} ({cds.length:5d}bp) {cds.gene or cds.locus_tag:20s} {cds.product[:40]}")
else:
    print("\n✅ 所有 Prokka CDS 都在 antiSMASH 中找到了匹配")

## 6. 按记录统计

In [ ]:
# 查看每个记录（contig/序列）的详细统计
print("按记录统计:")
print("=" * 80)
print(f"{'记录ID':<30} {'antiSMASH':>10} {'Prokka':>10} {'匹配':>10} {'仅AS':>10} {'仅PK':>10}")
print("-" * 80)

for record_id, stats in results['record_stats'].items():
    print(f"{record_id:<30} {stats['antismash_total']:>10} {stats['prokka_total']:>10} "
          f"{stats['matched']:>10} {stats['antismash_only']:>10} {stats['prokka_only']:>10}")

## 7. 导出结果到 CSV

In [ ]:
# 导出为 CSV 文件
output_prefix = Path('test_output/cds_comparison')
output_prefix.parent.mkdir(parents=True, exist_ok=True)

export_comparison_to_csv(results, output_prefix)

print("\n生成的文件:")
for csv_file in output_prefix.parent.glob(f"{output_prefix.name}*.csv"):
    print(f"  - {csv_file}")

## 8. 可视化比较（可选）

In [ ]:
# 可视化统计结果
try:
    import matplotlib.pyplot as plt
    
    summary = results['summary']
    
    # 饼图：CDS 匹配情况
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 左图：总体匹配情况
    labels = ['匹配', '仅 antiSMASH', '仅 Prokka']
    sizes = [
        summary['total_matched'],
        summary['total_antismash_only'],
        summary['total_prokka_only']
    ]
    colors = ['#2ecc71', '#3498db', '#e74c3c']
    
    ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax1.set_title('CDS 匹配情况', fontsize=14, fontweight='bold')
    
    # 右图：数量对比
    tools = ['antiSMASH', 'Prokka', '匹配']
    counts = [
        summary['total_antismash'],
        summary['total_prokka'],
        summary['total_matched']
    ]
    
    ax2.bar(tools, counts, color=['#3498db', '#e74c3c', '#2ecc71'])
    ax2.set_ylabel('CDS 数量', fontsize=12)
    ax2.set_title('CDS 数量对比', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    
    # 在柱状图上显示数值
    for i, v in enumerate(counts):
        ax2.text(i, v + max(counts) * 0.02, str(v), ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('test_output/cds_comparison_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✅ 可视化图表已保存到 test_output/cds_comparison_plot.png")
    
except ImportError:
    print("⚠️  需要安装 matplotlib 才能生成可视化图表")
    print("   安装命令: pip install matplotlib")

## 9. 总结和建议

### 如何解读结果

1. **匹配率 > 90%**: 说明两个工具注释结果高度一致
2. **匹配率 70-90%**: 存在一定差异，需要进一步分析
3. **匹配率 < 70%**: 差异较大，建议检查输入数据和参数设置

### 常见差异原因

- **最小 CDS 长度**: Prokka 和 antiSMASH 可能使用不同的最小长度阈值
- **基因预测算法**: 两者使用不同的基因查找工具
- **起始密码子选择**: 对于同一基因可能选择不同的起始位置
- **假基因处理**: antiSMASH 更专注于次级代谢产物相关基因

### 下一步分析

- 对于不匹配的 CDS，可以手动检查序列
- 考虑使用第三方工具（如 RAST, PGAP）进行交叉验证
- 关注特定功能基因（如抗生素合成基因簇）的一致性

## 附录：命令行使用方法

除了在 Notebook 中使用，也可以直接在命令行运行比较脚本：

```bash
# 比较单个文件
python compare_cds_annotations.py \
    --antismash path/to/antismash.gbk \
    --prokka path/to/prokka.gbk \
    --output cds_comparison.xlsx

# 比较目录
python compare_cds_annotations.py \
    --antismash-dir path/to/antismash_output \
    --prokka-dir path/to/prokka_output \
    --tolerance 15 \
    --show-details \
    --output cds_comparison.xlsx
```